# Chatbot de Atendimento - Revisa Master

Este notebook implementa um chatbot de atendimento ao cliente para a **Revisa Master** (www.revisamaster.com.br), especializado em qualificar leads frios vindos do Instagram e direcionar para agendamento via WhatsApp.

## Tecnicas de Prompt Engineering aplicadas

O chatbot aplica as tecnicas ensinadas nos notebooks do curso:

| Tecnica | Aula | Aplicacao no chatbot |
|---------|------|---------------------|
| Controle de temperatura | l01 | Temperature 0.3 para tom natural sem alucinacao |
| Delimitadores | l02 | Base de conhecimento separada com `###` no system prompt |
| Saida estruturada (JSON) | l02 | Extracao de dados do lead em JSON |
| Few-shot | l02 | Exemplos de dialogos ideais no system prompt |
| Chain-of-thought | l02 | Fluxo de qualificacao passo a passo |
| Refinamento iterativo | l03 | System prompt refinado progressivamente |
| Inferencia | l05 | Classificacao automatica do lead |
| Tom e linguagem | l06 | Portugues acolhedor e profissional |
| Expansao personalizada | l07 | Respostas adaptadas ao perfil do lead |

**Modelo:** OpenAI GPT (gpt-4o-mini)

## Configuracao

Instale as dependencias e configure sua chave da API da OpenAI no arquivo `.env`:

```
pip install openai python-dotenv
```

No arquivo `.env`, adicione:
```
OPENAI_API_KEY=sk-...
```

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

## Funcoes auxiliares

Duas funcoes seguindo o padrao do curso:
- `get_completion(prompt)` - chamada single-turn (compativel com os notebooks anteriores)
- `get_chat_completion(messages)` - chamada multi-turn (core do chatbot)

In [ ]:
def get_completion(prompt, model="gpt-4o-mini", temperature=0):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def get_chat_completion(messages, model="gpt-4o-mini", temperature=0.3):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

## System Prompt - Persona e Base de Conhecimento

O system prompt define:
1. **Persona** - Ana, assistente virtual acolhedora e profissional
2. **Base de conhecimento** - servicos, equipe, contato (delimitada com `###`)
3. **Fluxo de qualificacao** - passo a passo (chain-of-thought)
4. **Few-shot** - exemplos de dialogos ideais
5. **Regras** - limites do que o chatbot pode e nao pode dizer

In [ ]:
SYSTEM_PROMPT = """
Voce e a Ana, assistente virtual da Revisa Master (www.revisamaster.com.br).
Voce atende leads que chegam pelo Instagram e seu objetivo e qualifica-los e
direcioná-los para agendamento de uma consultoria via WhatsApp.

### PERSONA ###
- Nome: Ana
- Tom: acolhedor, empatico, profissional, nunca agressivo ou insistente
- Linguagem: portugues brasileiro, tratamento por "voce", informal mas profissional
- Voce entende o estresse academico e demonstra empatia genuina
- Respostas curtas e objetivas (maximo 3-4 frases por mensagem)

### BASE DE CONHECIMENTO ###
Servicos oferecidos pela Revisa Master:
1. Revisao ABNT - revisao completa de normas ABNT para dissertacoes, teses e TCCs
2. Formatacao academica - ajustes de layout, templates, normas tecnicas
3. Mentoria academica - orientacao personalizada para mestrandos e doutorandos
4. Suporte a escrita - desenvolvimento de texto, coesao, coerencia, argumentacao
5. Storytelling academico - construcao de narrativa clara e impactante para a pesquisa

Sobre a empresa:
- Equipe formada por mestres e doutores com experiencia academica
- Diferencial: suporte premium e direcionamento claro que a academia nem sempre oferece
- Orcamento personalizado entregue em ate 1 dia util
- Contato: WhatsApp +55 19 99539-2568 | Email: a.revisamaster@gmail.com
- Site: www.revisamaster.com.br

### FLUXO DE QUALIFICACAO (siga estes passos na ordem) ###
1. Cumprimente o lead de forma acolhedora. Se ele disser o nome, use-o.
2. Pergunte em que etapa academica ele esta (mestrado, doutorado, TCC, pos-graduacao).
3. Identifique a necessidade especifica (revisao, formatacao, mentoria, escrita, storytelling).
4. Pergunte sobre urgencia e prazo de entrega.
5. Apresente o servico mais relevante com base nas respostas.
6. Direcione para o WhatsApp (+55 19 99539-2568) para receber um orcamento personalizado.

Nao pule etapas. Faca uma pergunta por vez para nao sobrecarregar o lead.

### EXEMPLOS DE DIALOGOS ###

Exemplo 1:
Lead: "Oi, vi voces no Instagram. Preciso de ajuda com minha dissertacao."
Ana: "Oi! Que bom que nos encontrou! Fico feliz em ajudar. Me conta, voce esta no mestrado ou doutorado?"
Lead: "Mestrado em administracao."
Ana: "Que legal! E qual e a sua maior dificuldade agora? Precisa de revisao ABNT, ajuda com a escrita, ou orientacao geral sobre a dissertacao?"
Lead: "Meu orientador pediu pra revisar toda a formatacao ABNT."
Ana: "Entendo, isso e bem comum! Nossa equipe de mestres e doutores faz a revisao ABNT completa. Voce tem prazo pra entregar?"
Lead: "Preciso entregar em 2 semanas."
Ana: "Da tempo sim! Para te passar um orcamento personalizado, me chama no WhatsApp: +55 19 99539-2568. A equipe responde em ate 1 dia util. Posso ajudar em mais alguma coisa?"

Exemplo 2:
Lead: "Quanto custa a revisao?"
Ana: "O valor depende do tamanho e tipo do trabalho, por isso fazemos orcamentos personalizados. Me conta um pouco mais: voce esta em que etapa academica e qual o tipo de trabalho?"

### REGRAS ###
- NUNCA invente precos ou valores. Sempre direcione para orcamento personalizado.
- NUNCA prometa prazos especificos de entrega sem consulta da equipe.
- Se o lead perguntar algo fora do escopo (ex: conteudo tecnico da pesquisa), diga que
  a equipe de mentores pode ajudar e direcione para o WhatsApp.
- Se o lead nao quiser informar dados, respeite e ofereca o WhatsApp como alternativa.
- Sempre encerre com uma opcao de proximo passo (WhatsApp, email ou site).
"""

## Gerenciamento de conversa

A API da OpenAI e stateless - nao guarda historico entre chamadas. Por isso,
mantemos uma lista `conversation_history` que acumula todas as mensagens
(system + user + assistant) e enviamos o historico completo a cada chamada.

In [ ]:
conversation_history = []


def reset_conversation():
    """Reinicia a conversa com apenas o system prompt."""
    global conversation_history
    conversation_history = [{"role": "system", "content": SYSTEM_PROMPT}]


def chat(user_message):
    """Envia mensagem do usuario e retorna resposta da Ana."""
    conversation_history.append({"role": "user", "content": user_message})
    response = get_chat_completion(conversation_history)
    conversation_history.append({"role": "assistant", "content": response})
    return response


def show_history():
    """Exibe o historico da conversa (para debug)."""
    for msg in conversation_history:
        if msg["role"] == "system":
            continue
        role = "Lead" if msg["role"] == "user" else "Ana"
        print(f"{role}: {msg['content']}\n")


# Inicializa a conversa
reset_conversation()

## Demonstracao 1 - Mestrando precisando de revisao ABNT

Conversa simulada com um lead tipico: mestrando que viu o perfil no Instagram
e precisa de revisao de formatacao.

In [ ]:
reset_conversation()

mensagens_lead_1 = [
    "Oi, vim pelo Instagram de voces. Preciso de uma ajuda com minha dissertacao.",
    "To no mestrado em educacao, segundo ano ja.",
    "Preciso de revisao ABNT, meu orientador reclamou da formatacao toda.",
    "Tenho que entregar em 3 semanas.",
]

for msg in mensagens_lead_1:
    print(f"Lead: {msg}")
    response = chat(msg)
    print(f"Ana: {response}")
    print("---")

## Demonstracao 2 - Aluno de TCC buscando mentoria

Conversa simulada com um perfil diferente: aluno de graduacao com dificuldade
na escrita do TCC.

In [ ]:
reset_conversation()

mensagens_lead_2 = [
    "Boa tarde! Vi um post de voces sobre mentoria academica.",
    "Sou da graduacao, to fazendo meu TCC.",
    "To perdido na escrita, nao sei como organizar os capitulos.",
    "Minha defesa e em 2 meses.",
]

for msg in mensagens_lead_2:
    print(f"Lead: {msg}")
    response = chat(msg)
    print(f"Ana: {response}")
    print("---")

## Teste interativo

Execute a celula abaixo para conversar com a Ana em tempo real.
Digite `sair` para encerrar.

In [ ]:
reset_conversation()
print("=== Chatbot Revisa Master ===")
print("Digite 'sair' para encerrar.\n")

while True:
    user_input = input("Voce: ")
    if user_input.lower().strip() == "sair":
        print("\nConversa encerrada. Obrigado!")
        break
    response = chat(user_input)
    print(f"Ana: {response}\n")

## Extracao de dados do lead (Structured Output)

Apos a conversa, usamos a tecnica de **saida estruturada** (l02) para extrair
os dados do lead em formato JSON. Isso permite alimentar um CRM ou planilha
de acompanhamento.

In [ ]:
def extract_lead_info():
    """Extrai informacoes do lead a partir do historico da conversa."""
    # Monta o texto da conversa (sem o system prompt)
    conversa_texto = ""
    for msg in conversation_history:
        if msg["role"] == "system":
            continue
        role = "Lead" if msg["role"] == "user" else "Ana"
        conversa_texto += f"{role}: {msg['content']}\n"

    prompt = f"""
Analise a conversa abaixo delimitada por ``` e extraia as informacoes do lead
em formato JSON com exatamente estas chaves:

- nome: nome do lead (ou "nao informado")
- etapa_academica: mestrado, doutorado, TCC, pos-graduacao ou "nao informado"
- necessidade: revisao_abnt, formatacao, mentoria, escrita, storytelling ou "nao informado"
- urgencia: alta (menos de 2 semanas), media (2-4 semanas), baixa (mais de 1 mes) ou "nao informado"
- servico_recomendado: o servico mais adequado da Revisa Master
- qualificado: true se o lead demonstrou interesse real e informou necessidade, false caso contrario
- proximo_passo: a acao recomendada (ex: "enviar orcamento via WhatsApp")

Retorne APENAS o JSON, sem texto adicional.

Conversa:
```
{conversa_texto}
```
"""
    response = get_completion(prompt)
    try:
        # Remove possivel markdown de code block
        clean = response.strip()
        if clean.startswith("```"):
            clean = clean.split("\n", 1)[1]
            clean = clean.rsplit("```", 1)[0]
        return json.loads(clean)
    except json.JSONDecodeError:
        return {"erro": "Nao foi possivel extrair JSON", "resposta_bruta": response}


# Extrai dados do ultimo lead da conversa ativa
lead_info = extract_lead_info()
print(json.dumps(lead_info, indent=2, ensure_ascii=False))

## Conclusao

### Tecnicas aplicadas do curso

| Aula | Tecnica | Como foi aplicada |
|------|---------|-------------------|
| l01 | Temperatura | 0.3 para tom conversacional natural |
| l02 | Delimitadores | `###` separando secoes do system prompt, ``` na extracao |
| l02 | Saida estruturada | JSON na extracao de dados do lead |
| l02 | Few-shot | Exemplos de dialogos no system prompt |
| l02 | Chain-of-thought | Fluxo de qualificacao em 6 passos |
| l03 | Refinamento iterativo | System prompt pode ser ajustado progressivamente |
| l05 | Inferencia | Classificacao automatica do lead (qualificado/nao) |
| l06 | Tom e linguagem | Portugues acolhedor e profissional |
| l07 | Expansao | Respostas personalizadas ao perfil do lead |

### Proximos passos

1. **Integracao com WhatsApp** - conectar via API do WhatsApp Business
2. **Persistencia** - salvar conversas e dados dos leads em banco de dados
3. **Deploy** - transformar em aplicacao web (Streamlit/Gradio) ou API REST
4. **Metricas** - acompanhar taxa de qualificacao e conversao